# Valor y fuga de clientes

RFM de 180 días y antigüedad de registro MongoDB; objetivos a 90 días. Cohortes cronológicas con etiquetas maduras. Random Forest compite con una referencia poblacional; se selecciona por RMSE o Brier en validación y se informa un test temporal independiente.

Las funciones de producción son la única implementación del entrenamiento. Estos experimentos registran artefactos en MLflow; no sustituyen la publicación activa. El DAG publica los siete módulos de forma atómica. No se consultan datos personales de contacto.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'hgc-ml').exists())
sys.path.insert(0, str(ROOT / 'hgc-ml'))
from core.settings import postgres_engine
from training.common import setup_tracking
engine = postgres_engine()
setup_tracking()
with engine.connect() as connection:
    load_id = connection.execute(text('SELECT load_id FROM hgc_analytics.loads ORDER BY loaded_at DESC LIMIT 1')).scalar_one()
print('Snapshot:', load_id)


Snapshot: befab29d-54d0-4c88-aee6-0d59c6e120aa


In [2]:
frame=pd.read_sql_table('sys_customer_features',engine,schema='hgc_analytics')
frame.groupby('cutoff').agg(customers=('id_cliente','size'),mature_labels=('spend_90d','count'))

,customers,mature_labels
cutoff,,
2025-01-12,119619,119619
2025-04-12,118389,118389
2025-07-11,117744,117744
2025-10-09,116378,116378
2026-01-07,116302,116302
2026-04-07,117979,117979
2026-07-06,118659,118659
2026-10-04,118498,0


In [3]:
from training.customers import train_customers
reports=train_customers(frame,load_id)
pd.DataFrame([{ 'module':name, **report['model']['metrics'], 'algorithm':report['model']['algorithm'],'run_id':report['model']['run_id']} for name,report in reports.items()])

2026/10/05 03:54:28 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/05 03:54:29 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


2026/10/05 03:54:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'hgc_clv' already exists. Creating a new version of this model...
Created version '7' of model 'hgc_clv'.


2026/10/05 03:54:41 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


/usr/local/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


2026/10/05 03:54:41 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: 'getpwuid(): uid not found: 50000'


2026/10/05 03:54:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered model 'hgc_churn' already exists. Creating a new version of this model...
Created version '6' of model 'hgc_churn'.


,module,rmse_bob,validation_candidate,validation_baseline,test_customers,mae_bob,algorithm,run_id,brier,roc_auc
0,clv,143.749599,141.448608,141.387838,118659,82.577307,Ingreso medio validado,2b688f0bca0e4b198e894590fe670311,NaN,NaN
1,churn,NaN,0.171769,0.171630,118659,NaN,Tasa poblacional validada,488c462ce8dc472cbdd1a8208473130e,0.172474,0.5


In [4]:
for name, report in reports.items():
    print(name, report['model']['limitations'])
assert all(len(report['rows']) > 0 for report in reports.values())
del frame, reports

clv Ingreso esperado a 90 días, no utilidad ni valor de vida completo. Las variables disponibles no superaron la referencia poblacional. La estimación no discrimina entre clientes; no usarla como ranking individual.
churn Probabilidad de no comprar durante 90 días. No implica baja definitiva; umbral operativo de 0,5. Las variables disponibles no superaron la referencia poblacional. La estimación no discrimina entre clientes; no usarla como ranking individual.
